In [1]:
import re
import pickle
import pandas as pd
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer

# ============================================================
# KONFIGURASI — sesuaikan kalau perlu
# ============================================================
TRAIN_CSV       = 'youtube_enriched_cluster.csv'
TOKENIZER_PATH  = 'tokenizer.pkl'
MAX_NUM_WORDS   = 10000
# ============================================================

# Pastikan stopwords bahasa Indonesia sudah tersedia
try:
    stop_words = set(stopwords.words('indonesian'))
except LookupError:
    import nltk
    nltk.download('stopwords')
    stop_words = set(stopwords.words('indonesian'))


def clean_text(text):
    """Bersihkan teks judul: lowercase, hapus non-huruf, hapus stopwords & kata pendek."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join([w for w in text.split() if w not in stop_words and len(w) > 1])
    return text


def build_tokenizer(csv_path=TRAIN_CSV, max_num_words=MAX_NUM_WORDS):
    print(f"📂 Membaca data dari '{csv_path}'...")
    df = pd.read_csv(csv_path)

    if 'Title' not in df.columns:
        # fallback kalau nama kolomnya lowercase
        if 'title' in df.columns:
            df.rename(columns={'title': 'Title'}, inplace=True)
        else:
            raise KeyError("Kolom 'Title' tidak ditemukan di CSV. Cek nama kolomnya.")

    print("🧹 Membersihkan teks judul...")
    df['clean_title'] = df['Title'].apply(clean_text)
    df = df[df['clean_title'].str.len() > 0].copy()
    print(f"✅ Jumlah judul valid setelah cleaning: {len(df)}")

    print("🔧 Fitting tokenizer...")
    tokenizer = Tokenizer(num_words=max_num_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(df['clean_title'])
    print(f"✅ Tokenizer selesai dibuat — vocab size: {len(tokenizer.word_index)}")

    return tokenizer


def save_tokenizer(tokenizer, path=TOKENIZER_PATH):
    with open(path, 'wb') as f:
        pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"💾 Tokenizer disimpan ke '{path}'")


if __name__ == '__main__':
    tok = build_tokenizer()
    save_tokenizer(tok)

    # Tes cepat: tampilkan 10 kata teratas di vocab
    print("\n🔎 Contoh 10 kata teratas dalam vocab:")
    for word, idx in list(tok.word_index.items())[:10]:
        print(f"   {idx}: {word}")

📂 Membaca data dari 'youtube_enriched_cluster.csv'...
🧹 Membersihkan teks judul...
✅ Jumlah judul valid setelah cleaning: 1032
🔧 Fitting tokenizer...
✅ Tokenizer selesai dibuat — vocab size: 2820
💾 Tokenizer disimpan ke 'tokenizer.pkl'

🔎 Contoh 10 kata teratas dalam vocab:
   1: <OOV>
   2: raymondchin
   3: narasi
   4: indonesia
   5: daily
   6: azka
   7: shorts
   8: live
   9: corbuzier
   10: dunia
